In [ ]:
!pip install sentence-transformers wandb -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb

In [ ]:
wandb.login()

wandb.init(
    project="23f1000054-t22026",
    name="mlp_minilm_v1"
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
train = pd.read_csv("/content/drive/MyDrive/SmartMCQ/train.csv")
test = pd.read_csv("/content/drive/MyDrive/SmartMCQ/test.csv")
print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [ ]:
rows = []
for _, row in train.iterrows():
    prompt = row["prompt"]
    answer = row["answer"]
    for option in ["A","B","C","D","E"]:
        rows.append({
            "text": prompt + " [SEP] " + str(row[option]),
            "label": int(option == answer)
        })
binary_train = pd.DataFrame(rows)
print(binary_train.shape)
binary_train.head()

(10000, 2)


,text,label
0,Pick the best possible answer: What is Martin ...,0
1,Pick the best possible answer: What is Martin ...,1
2,Pick the best possible answer: What is Martin ...,0
3,Pick the best possible answer: What is Martin ...,0
4,Pick the best possible answer: What is Martin ...,0


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    binary_train["text"].tolist(),
    batch_size=64,
    show_progress_bar=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    embeddings,
    binary_train["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=binary_train["label"]
)
print(X_train.shape)
print(X_valid.shape)

(8000, 384)
(2000, 384)


In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_valid = torch.tensor(y_valid, dtype=torch.float32)

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_dataset = MCQDataset(X_train, y_train)
valid_dataset = MCQDataset(X_valid, y_valid)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=128,
    shuffle=False
)

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(384, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.network(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MLPClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch).squeeze()
        loss = criterion(
            outputs,
            y_batch
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    model.eval()
    preds = []
    truths = []
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch).squeeze()
            probs = torch.sigmoid(outputs)
            pred = (probs > 0.5).cpu().numpy()
            preds.extend(pred)
            truths.extend(
                y_batch.numpy()
            )
    acc = accuracy_score(
        truths,
        preds
    )
    f1 = f1_score(
        truths,
        preds
    )
    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss={avg_loss:.4f} "
        f"Acc={acc:.4f} "
        f"F1={f1:.4f}"
    )
    wandb.log({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "accuracy": acc,
        "f1": f1
    })

Epoch 1/10 Loss=0.5189 Acc=0.8000 F1=0.0000
Epoch 2/10 Loss=0.4755 Acc=0.8000 F1=0.0000
Epoch 3/10 Loss=0.4429 Acc=0.8060 F1=0.0673
Epoch 4/10 Loss=0.4081 Acc=0.8415 F1=0.4053
Epoch 5/10 Loss=0.3745 Acc=0.8590 F1=0.4854
Epoch 6/10 Loss=0.3393 Acc=0.8555 F1=0.5191
Epoch 7/10 Loss=0.3300 Acc=0.8690 F1=0.5387
Epoch 8/10 Loss=0.3003 Acc=0.8730 F1=0.5836
Epoch 9/10 Loss=0.2983 Acc=0.8840 F1=0.6054
Epoch 10/10 Loss=0.2846 Acc=0.8845 F1=0.6351


In [ ]:
torch.save(
    model.state_dict(),
    "mlp_model.pt"
    )

print("Model Saved")

Model Saved


In [ ]:
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/SmartMCQ/mlp_model.pt"
)

In [ ]:
wandb.finish()

accuracy,▁▁▁▄▆▆▇▇██
epoch,▁▂▃▃▄▅▆▆▇█
f1,▁▁▂▅▆▇▇▇██
loss,█▇▆▅▄▃▂▁▁▁
accuracy,0.8845
epoch,10
f1,0.63507
loss,0.28457
